![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Proyecto 2 - Clasificación de género de películas

El propósito de este proyecto es que puedan poner en práctica, en sus respectivos grupos de trabajo, sus conocimientos sobre técnicas de preprocesamiento, modelos predictivos de NLP, y la disponibilización de modelos. Para su desarrollo tengan en cuenta las instrucciones dadas en la "Guía del proyecto 2: Clasificación de género de películas"

**Entrega**: La entrega del proyecto deberán realizarla durante la semana 8. Sin embargo, es importante que avancen en la semana 7 en el modelado del problema y en parte del informe, tal y como se les indicó en la guía.

Para hacer la entrega, deberán adjuntar el informe autocontenido en PDF a la actividad de entrega del proyecto que encontrarán en la semana 8, y subir el archivo de predicciones a la [competencia de Kaggle](https://www.kaggle.com/t/2c54d005f76747fe83f77fbf8b3ec232).

## Datos para la predicción de género en películas

![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/moviegenre.png)

En este proyecto se usará un conjunto de datos de géneros de películas. Cada observación contiene el título de una película, su año de lanzamiento, la sinopsis o plot de la película (resumen de la trama) y los géneros a los que pertenece (una película puede pertenercer a más de un género). Por ejemplo:
- Título: 'How to Be a Serial Killer'
- Plot: 'A serial killer decides to teach the secrets of his satisfying career to a video store clerk.'
- Generos: 'Comedy', 'Crime', 'Horror'

La idea es que usen estos datos para predecir la probabilidad de que una película pertenezca, dada la sinopsis, a cada uno de los géneros.

Agradecemos al profesor Fabio González, Ph.D. y a su alumno John Arevalo por proporcionar este conjunto de datos. Ver https://arxiv.org/abs/1702.01992

## Ejemplo predicción conjunto de test para envío a Kaggle
En esta sección encontrarán el formato en el que deben guardar los resultados de la predicción para que puedan subirlos a la competencia en Kaggle.

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Importación librerías
import pandas as pd
import os
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier,BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D, Input, Concatenate,GlobalMaxPooling1D,LSTM,Dropout
from keras.models import Model



# Descargar recursos de nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ariel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ariel\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ariel\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
# Carga de datos de archivo .csv
dataTraining = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)
dataTesting = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTesting.zip', encoding='UTF-8', index_col=0)

In [4]:
# Visualización datos de entrenamiento
dataTraining.head()
dataTraining.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7895 entries, 3107 to 215
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    7895 non-null   int64  
 1   title   7895 non-null   object 
 2   plot    7895 non-null   object 
 3   genres  7895 non-null   object 
 4   rating  7895 non-null   float64
dtypes: float64(1), int64(1), object(3)
memory usage: 370.1+ KB


In [5]:
# Visualización datos de test
dataTesting.head()
dataTesting.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3383 entries, 1 to 11275
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   year    3383 non-null   int64 
 1   title   3383 non-null   object
 2   plot    3383 non-null   object
dtypes: int64(1), object(2)
memory usage: 105.7+ KB


In [6]:
# Definición de variables predictoras (X)
vect = CountVectorizer(max_features=1000)
X_dtm = vect.fit_transform(dataTraining['plot'])
X_dtm.shape

(7895, 1000)

## Preprocesamiento de datos

In [7]:
# Definición de variable de interés (y)
dataTraining['genres'] = dataTraining['genres'].map(lambda x: eval(x))
le = MultiLabelBinarizer()
y_genres = le.fit_transform(dataTraining['genres'])

Se preprocesan las variables eliminando stop words, lematizando y transformando todo el texto a minusculas. Adicionalmente se decide unir el titulo y trama en una sola variable.

In [8]:
# Eliminar la columna 'ratings' del conjunto de datos de entrenamiento
dataTraining.drop(columns=['rating'], inplace=True)


# Limpiar y normalizar el texto
def preprocess_text(text):
    # Eliminar caracteres especiales y convertir a minúsculas
    text = re.sub(r'\W', ' ', text)
    text = text.lower()

    # Eliminar stopwords y lematizar
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    text = ' '.join([lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words])
    return text
#Creación de una nueva variable llamada plot_2 para unir las varibles "title" y "plot"
dataTraining['plot_2'] = dataTraining['title'] + ", " + dataTraining['plot']
dataTesting['plot_2'] = dataTesting['title'] + ", " + dataTesting['plot']
dataTraining['plot_2'] = dataTraining['plot_2'].apply(preprocess_text)
dataTesting['plot_2'] = dataTesting['plot_2'].apply(preprocess_text)

In [9]:
# Visualización datos de entrenamiento
dataTesting.head()

,year,title,plot,plot_2
1,1999,Message in a Bottle,"who meets by fate , shall be sealed by fate ....",message bottle meet fate shall sealed fate the...
4,1978,Midnight Express,"the true story of billy hayes , an american c...",midnight express true story billy hayes americ...
5,1996,Primal Fear,martin vail left the chicago da ' s office to ...,primal fear martin vail left chicago da office...
6,1950,Crisis,husband and wife americans dr . eugene and mr...,crisis husband wife american dr eugene mr hele...
7,1959,The Tingler,the coroner and scientist dr . warren chapin ...,tingler coroner scientist dr warren chapin res...


In [10]:
dataTraining['plot_2'].str.split().str.len().describe()

count    7895.000000
mean       70.676377
std        44.528604
min         1.000000
25%        40.000000
50%        62.000000
75%        89.000000
max       822.000000
Name: plot_2, dtype: float64

Se realiza la tokenizacion considerando 10000 palabras, esta cantidad representa más del 90% de las palabras totales.

Para la secuenciación se toma como largo máximo del texto 100 palabras porque el 75% de los datos tienen menos de 90 palabras.

In [11]:
# Tokenización y secuenciación del texto
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(dataTraining['plot_2'])

word_counts = tokenizer.word_counts

tot_word_occurrences = sum(word_counts.values())

srt_word_counts = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)

sel_words_count = sum([count for word, count in srt_word_counts[:10000]])

pct = (sel_words_count / tot_word_occurrences) * 100

print(f'10000 palabras corresponden al {pct}% del total de palabras')
X_train_seq = tokenizer.texts_to_sequences(dataTraining['plot_2'])
X_train_pad = pad_sequences(X_train_seq, maxlen=100,truncating='post',padding="post")

X_test_seq = tokenizer.texts_to_sequences(dataTesting['plot_2'])
X_test_pad = pad_sequences(X_test_seq, maxlen=100,truncating='post',padding="post")

10000 palabras corresponden al 90.98872739654833% del total de palabras


## Modelos

### Vectorizacion

Se aplica vectorizacion con TFIDF para generar valores numericos a partir del texto en base a la frecuencia de cada palabra tanto dentro del texto como en todo el conjunto de textos. Posteriormente se realizan estimaciones con un modelo de regresión logistica, random forest y xgboost.

In [12]:
X_train, X_test, y_train_genres, y_test_genres = train_test_split(dataTraining['plot_2'], y_genres, test_size=0.2, random_state=42)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [14]:
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X_train_tfid = vectorizer.fit_transform(X_train.tolist())
X_test_tfid = vectorizer.transform(X_test.tolist())
X_sub_tfid = vectorizer.transform(dataTesting['plot_2'].tolist())

#### Regresion Logistica

In [ ]:
log_clf=OneVsRestClassifier(LogisticRegression(max_iter=100000))
log_clf.fit(X_train_tfid, y_train_genres)

OneVsRestClassifier(estimator=LogisticRegression(max_iter=100000))

In [ ]:
y_test_proba = log_clf.predict(X_test_tfid)
roc_auc_score(y_test_genres, y_test_proba, average='macro')

np.float64(0.57625708562646)

#### Random Forest

In [ ]:
# Definición y entrenamiento
rf_clf = OneVsRestClassifier(RandomForestClassifier(n_jobs=-1, n_estimators=100, max_depth=30, random_state=42,max_features=0.5))
rf_clf.fit(X_train_tfid, y_train_genres)

OneVsRestClassifier(estimator=RandomForestClassifier(max_depth=30,
                                                     max_features=0.5,
                                                     n_jobs=-1,
                                                     random_state=42))

In [ ]:
y_test_proba = rf_clf.predict(X_test_tfid)
roc_auc_score(y_test_genres, y_test_proba, average='macro')

np.float64(0.5848905930528973)

#### XGBOOST

In [19]:
from xgboost import XGBClassifier

In [ ]:
xgb_clf = OneVsRestClassifier(XGBClassifier(n_jobs=-1, n_estimators=100, max_depth=10, random_state=0,learning_rate=0.1,subsample=0.5, colsample_bytree=0.5))
xgb_clf.fit(X_train_tfid, y_train_genres)

OneVsRestClassifier(estimator=XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.5, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=0.1, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=10,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=100, n_jobs=-1,
                                            num_parallel_tree=None, ...))

In [ ]:
y_test_proba = xgb_clf.predict(X_test_tfid)
roc_auc_score(y_test_genres, y_test_proba, average='macro')

np.float64(0.5995634403205492)

### Embedding propio

En busca de un mejor rendimiento se genera un embedding propio para poder capturar el significado de cada palabra presente en la trama.

In [22]:

# División de los datos en conjuntos de entrenamiento y validación
X_train, X_test, y_train_genres, y_test_genres = train_test_split(X_train_pad, y_genres, test_size=0.2, random_state=42)


In [23]:
from keras.metrics import AUC

Se utiliza como parametro de early stopping el auc del conjunto de validacion, con una paciencia de 5. Es decir, si el auc no aumenta tras 5 epocas mantenemos el modelo con mejor auc y detenemos el entrenamiento.

In [24]:
early_stop = EarlyStopping(
    monitor='val_auc',
    patience=5,
    restore_best_weights=True
)

Para la estimacion se utiliza una red neuronal, donde se crea  un embedding con los inputs definidos previamente (10000 palabras posibles y secuencias de 100 palabras) y un output de 128 para el vector de embedding. Posterior se obtiene el promedio de cada observacion para capturar su significado.

Se utilizaron dos modelos uno con una capa 'relu' oculta y una 'sigmoide' final y otro solo con la capa 'sigmoide' final. Todos los modelos usan el criterio de early stopping definido preivamente.

#### Red con capa oculta

In [ ]:
n_classes=len(le.classes_)

hl_model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=100),
    GlobalAveragePooling1D(),
    Dense(64, activation='relu'),
    Dense(n_classes, activation='sigmoid')
])

hl_model.compile(optimizer='adam',
              loss='binary_crossentropy', metrics=[AUC(multi_label=True, name='auc')])


In [ ]:
hl_model.fit(X_train,y_train_genres,epochs=50,verbose=1,validation_data=(X_test,y_test_genres),callbacks=[early_stop])

Epoch 1/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - auc: 0.4863 - loss: 0.4335 - val_auc: 0.5030 - val_loss: 0.3010
Epoch 2/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.5163 - loss: 0.2944 - val_auc: 0.5911 - val_loss: 0.2923
Epoch 3/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.6314 - loss: 0.2830 - val_auc: 0.6747 - val_loss: 0.2821
Epoch 4/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.7259 - loss: 0.2659 - val_auc: 0.7337 - val_loss: 0.2636
Epoch 5/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.7714 - loss: 0.2462 - val_auc: 0.7661 - val_loss: 0.2535
Epoch 6/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.8152 - loss: 0.2234 - val_auc: 0.7938 - val_loss: 0.2413
Epoch 7/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.8505 - loss: 0.2083 - val_auc: 0.8118 - val_loss: 0.2345
Epoch 8/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.8829 - loss: 0.1901 - val_auc: 0.8216 - val_loss: 0.2305
Epoch 9/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - au

In [ ]:
y_test_proba = hl_model.predict(X_test)
roc_auc_score(y_test_genres, y_test_proba, average='macro')

50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


np.float64(0.854449237087514)

#### Red sin capa oculta

In [ ]:
nn_model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=100),
    GlobalAveragePooling1D(),
    Dense(n_classes, activation='sigmoid')
])

nn_model.compile(optimizer='adam',
              loss='binary_crossentropy', metrics=[AUC(multi_label=True, name='auc')])

In [ ]:
nn_model.fit(X_train,y_train_genres,epochs=100,verbose=1,validation_data=(X_test,y_test_genres),callbacks=[early_stop])

Epoch 1/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - auc: 0.4943 - loss: 0.4375 - val_auc: 0.4959 - val_loss: 0.3072
Epoch 2/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.5051 - loss: 0.2981 - val_auc: 0.5326 - val_loss: 0.2974
Epoch 3/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.5570 - loss: 0.2900 - val_auc: 0.5893 - val_loss: 0.2913
Epoch 4/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.6434 - loss: 0.2826 - val_auc: 0.6465 - val_loss: 0.2846
Epoch 5/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.6837 - loss: 0.2708 - val_auc: 0.6901 - val_loss: 0.2769
Epoch 6/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.7348 - loss: 0.2626 - val_auc: 0.7283 - val_loss: 0.2689
Epoch 7/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.7807 - loss: 0.2505 - val_auc: 0.7601 - val_loss: 0.2603
Epoch 8/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - auc: 0.8184 - loss: 0.2381 - val_auc: 0.7828 - val_loss: 0.2526
Epoch 9/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/

In [ ]:
y_test_proba = nn_model.predict(X_test)
roc_auc_score(y_test_genres, y_test_proba, average='macro')

50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 938us/step


np.float64(0.8782397274772425)

#### Red con LSTM

Al sacar el promedio del vector de embedding se pierde el contexto adicional que otorga la secuencialidad de las palabras. Por ello, Tambien se aplica una red con LSTM para intentar capturar de mejor forma el siginificado de cada texto.

Se agrega una capa de Dropout para evitar el overfitting del modelo.

In [ ]:
lstm_model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=100),
    LSTM(64),
    Dropout(0.75),
    Dense(n_classes, activation='sigmoid')
])

lstm_model.compile(optimizer='adam',
              loss='binary_crossentropy', metrics=[AUC(multi_label=True, name='auc')])

In [ ]:
lstm_model.fit(X_train,y_train_genres,epochs=50,verbose=1,validation_data=(X_test,y_test_genres),callbacks=[early_stop])

Epoch 1/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - auc: 0.4980 - loss: 0.4560 - val_auc: 0.5104 - val_loss: 0.2980
Epoch 2/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - auc: 0.5017 - loss: 0.3237 - val_auc: 0.5169 - val_loss: 0.2971
Epoch 3/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - auc: 0.4900 - loss: 0.3103 - val_auc: 0.5005 - val_loss: 0.2976
Epoch 4/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - auc: 0.4958 - loss: 0.3075 - val_auc: 0.5254 - val_loss: 0.2973
Epoch 5/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - auc: 0.5189 - loss: 0.3017 - val_auc: 0.5286 - val_loss: 0.2966
Epoch 6/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - auc: 0.5403 - loss: 0.2994 - val_auc: 0.5392 - val_loss: 0.2959
Epoch 7/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - auc: 0.5533 - loss: 0.2970 - val_auc: 0.5827 - val_loss: 0.2914
Epoch 8/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - auc: 0.5835 - loss: 0.2892 - val_auc: 0.6068 - val_loss: 0.2830
Epoch 9/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/

In [33]:
y_test_proba = model.predict(X_test)
roc_auc_score(y_test_genres, y_test_proba, average='macro')

50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step


np.float64(0.7592848465818466)

### Transfer Learning

Se utiliza transfer learning para aplicar embedding a las observaciones con modelos previamente entrenados con bases de datos mucho más grandes.

Se decide utilizar el transformador "all-mpnet-base-v2" entrenado con un billón de pares de oraciones provenientes de fuentes como reddit, wikipedia, stack exchange, etc.

Dado que este transformador es capaz de capturar el contexto de las palabras se decide crear una nueva variable que recopile el año, título y trama de la pélicula.

In [34]:
dataTraining['wh_txt']='Year: '+dataTraining['year'].astype(str)+'; Title:'+dataTraining['title']+'; Plot:'+dataTraining['plot']
dataTesting['wh_txt']='Year: '+dataTesting['year'].astype(str)+'; Title:'+dataTesting['title']+'; Plot:'+dataTesting['plot']

In [35]:
X_train, X_test, y_train_genres, y_test_genres = train_test_split(dataTraining['wh_txt'], y_genres, test_size=0.2, random_state=42)

In [36]:
from sentence_transformers import SentenceTransformer


encoder = SentenceTransformer("all-mpnet-base-v2")


X_train_embeddings = encoder.encode(X_train.tolist(), show_progress_bar=True)
X_test_embeddings = encoder.encode(X_test.tolist(), show_progress_bar=True)
X_sub_embeddings = encoder.encode(dataTesting['wh_txt'].tolist(), show_progress_bar=True)

Batches: 100%|██████████| 106/106 [08:54<00:00,  5.04s/it]


Al igual que las redes neuronales anteriores se entrena una red con una capa oculta relu y una final sigmoid; y otra red solo con una capa final sigmoid. Asimismo, se utiliza el mismo parametro de early stopping y dropout para evitar el overfitting.

No se etrena una red con LSTM dado que el transformador utilizado ya considera el orden y contexto de las palabras.

#### Red con capa oculta

In [ ]:
tl_hl_model = Sequential([
    Dense(256, activation='relu', input_shape=(768,)),
    Dropout(0.5),
    Dense(n_classes, activation='sigmoid')
])

tl_hl_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[AUC(multi_label=True, name='auc')]
)

In [ ]:
tl_hl_model.fit(X_train_embeddings,y_train_genres,epochs=50,verbose=1,validation_data=(X_test_embeddings,y_test_genres),callbacks=[early_stop])

Epoch 1/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - auc: 0.5644 - loss: 0.3963 - val_auc: 0.8598 - val_loss: 0.2302
Epoch 2/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.8264 - loss: 0.2229 - val_auc: 0.9005 - val_loss: 0.2002
Epoch 3/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.8818 - loss: 0.2010 - val_auc: 0.8902 - val_loss: 0.1895
Epoch 4/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.8811 - loss: 0.1908 - val_auc: 0.8947 - val_loss: 0.1835
Epoch 5/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.8847 - loss: 0.1820 - val_auc: 0.8985 - val_loss: 0.1804
Epoch 6/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - auc: 0.9032 - loss: 0.1787 - val_auc: 0.9204 - val_loss: 0.1785
Epoch 7/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.9082 - loss: 0.1745 - val_auc: 0.9206 - val_loss: 0.1764
Epoch 8/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.9201 - loss: 0.1706 - val_auc: 0.9227 - val_loss: 0.1749
Epoch 9/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - au

In [ ]:
y_test_proba = tl_hl_model.predict(X_test_embeddings)
roc_auc_score(y_test_genres, y_test_proba, average='macro')

50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


np.float64(0.9263465473896891)

#### Red sin capa oculta

In [ ]:
tl_nn_model = Sequential([
    Dropout(0.5, input_shape=(768,)),
    Dense(n_classes, activation='sigmoid')
])

tl_nn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[AUC(multi_label=True, name='auc')]
)

In [ ]:
tl_nn_model.fit(X_train_embeddings,y_train_genres,epochs=50,verbose=1,validation_data=(X_test_embeddings,y_test_genres),callbacks=[early_stop])

Epoch 1/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - auc: 0.5575 - loss: 0.5457 - val_auc: 0.7392 - val_loss: 0.3352
Epoch 2/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7152 - loss: 0.3143 - val_auc: 0.7906 - val_loss: 0.2803
Epoch 3/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7663 - loss: 0.2719 - val_auc: 0.8235 - val_loss: 0.2587
Epoch 4/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.8027 - loss: 0.2508 - val_auc: 0.8468 - val_loss: 0.2452
Epoch 5/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.8186 - loss: 0.2416 - val_auc: 0.8627 - val_loss: 0.2353
Epoch 6/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.8455 - loss: 0.2310 - val_auc: 0.8750 - val_loss: 0.2275
Epoch 7/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.8594 - loss: 0.2230 - val_auc: 0.8831 - val_loss: 0.2211
Epoch 8/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.8640 - loss: 0.2166 - val_auc: 0.8893 - val_loss: 0.2158
Epoch 9/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - au

In [ ]:
y_test_proba = tl_nn_model.predict(X_test_embeddings)
roc_auc_score(y_test_genres, y_test_proba, average='macro')

50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


np.float64(0.919081285415912)

## Kaggle

In [ ]:
# transformación variables predictoras X del conjunto de test
X_test_dtm = vect.transform(dataTesting['plot_2'])

cols = ['p_Action', 'p_Adventure', 'p_Animation', 'p_Biography', 'p_Comedy', 'p_Crime', 'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History', 'p_Horror', 'p_Music', 'p_Musical', 'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport', 'p_Thriller', 'p_War', 'p_Western']

# Predicción del conjunto de test
#y_pred_test_genres = clf.predict_proba(X_test_pad)
y_pred_test_genres=tl_hl_model.predict(X_sub_embeddings)

106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 914us/step


In [44]:
# Guardar predicciones en formato exigido en la competencia de kaggle
res = pd.DataFrame(y_pred_test_genres, index=dataTesting.index, columns=cols)
res.to_csv('pred_genres_text_RF.csv', index_label='ID')
res.head()

,p_Action,p_Adventure,p_Animation,p_Biography,p_Comedy,p_Crime,p_Documentary,p_Drama,p_Family,p_Fantasy,...,p_Musical,p_Mystery,p_News,p_Romance,p_Sci-Fi,p_Short,p_Sport,p_Thriller,p_War,p_Western
1,0.036732,0.035306,0.003456,0.009903,0.417280,0.260594,0.000726,0.765783,0.013716,0.062482,...,0.002820,0.257500,0.000076,0.602834,0.013690,0.002834,0.003824,0.378555,0.000900,0.000535
4,0.055234,0.025153,0.001155,0.190635,0.064813,0.554776,0.025792,0.692550,0.005534,0.011515,...,0.007881,0.081649,0.000185,0.033710,0.005755,0.017913,0.003284,0.346718,0.008006,0.012993
5,0.051299,0.002339,0.000235,0.041119,0.042195,0.725144,0.004057,0.677936,0.000710,0.008489,...,0.000448,0.453016,0.000448,0.026190,0.021793,0.000649,0.001442,0.884522,0.002261,0.003338
6,0.015699,0.018136,0.000309,0.032104,0.095246,0.032441,0.000579,0.840562,0.001185,0.002536,...,0.005663,0.025175,0.000036,0.280240,0.074263,0.002260,0.000674,0.168398,0.058573,0.005124
7,0.012016,0.022483,0.006468,0.005269,0.099354,0.081471,0.000496,0.215186,0.029353,0.280800,...,0.007414,0.514026,0.000086,0.053881,0.496439,0.009620,0.000467,0.569411,0.000722,0.003491
